# Serpy — Advanced Tutorial Problems with Step-by-Step Solutions

This notebook is a second, independent advanced Serpy workbook.

It deliberately uses a **tutorial style** rather than a compact reference style. Most problems are broken into small stages:

1. understand the serialization requirement;
2. inspect the Python object shape;
3. choose the correct Serpy feature;
4. implement only one layer at a time;
5. serialize and inspect;
6. add assertions;
7. discuss design trade-offs and common mistakes.

The goal is not merely to make the code work. The goal is to understand **why a particular serializer design is appropriate**.


## What Serpy is good at

Serpy is a small, output-oriented serialization library.

Its main job is:

```text
Python objects / dictionaries
            |
            v
       Serpy serializer
            |
            v
   native Python data
   (dict / list / scalar)
            |
            v
   JSON, YAML, HTTP response,
   cache payload, log record, ...
```

Serpy's serializers are intentionally lightweight.

That also means we should avoid expecting features that belong to a full validation/deserialization framework.


## Important limitation

Serpy does **not** deserialize incoming dictionaries into domain objects, and its `data=` argument is not a validation API.

In this notebook, we will therefore separate two responsibilities:

- **input validation / object construction** happens before Serpy;
- **output projection / conversion** happens with Serpy.

Keeping that boundary clear prevents many architecture mistakes.


## Features used in this notebook

We will progressively use:

- `serpy.Serializer`
- `serpy.DictSerializer`
- `serpy.Field`
- `StrField`, `IntField`, `FloatField`, `BoolField`
- `attr=...`
- `label=...`
- `call=True`
- `required=False`
- nested serializers
- `many=True`
- `MethodField`
- custom `Field.to_value()`
- custom `Field.as_getter()`
- `getter_takes_serializer`
- serializer inheritance
- `.data` caching behavior
- JSON/YAML boundary encoding
- performance measurement

The examples are intentionally more realistic than a one-object demo.


## Setup

The examples target the public Serpy `0.3.1` API.

If Serpy is already available in your environment, the installation cell is harmless to skip.


In [ ]:
%pip install -q serpy==0.3.1 PyYAML


In [ ]:
from __future__ import annotations

import json
import operator
import timeit
from dataclasses import dataclass, field
from datetime import datetime, timezone
from decimal import Decimal, ROUND_HALF_UP
from enum import Enum
from typing import Any

import serpy
import yaml


## A tiny display helper

The helper below is not part of Serpy.

It only makes intermediate results easier to read while we work through the problems.


In [ ]:
def pretty(value):
    print(json.dumps(value, indent=2, sort_keys=True))


# Part 1 — Think of a serializer as an explicit public projection

A common beginner mistake is to think:

> "I have an object. How do I dump all of it?"

A better API-design question is:

> "Which parts of this object are intentionally public?"

Serpy works particularly well with the second mindset because every field is explicitly declared.


## Problem 1 — Project a messy internal object into a clean public shape

Imagine an internal `ServiceAccount` object.

It contains public data, operational data, and secrets.

We want an external API response containing only:

```json
{
  "id": 17,
  "name": "build-bot",
  "enabled": true
}
```

We must **not** expose the token or the internal retry counter.


### Step 1 — Define the domain object

Notice that the model is not designed specifically for serialization.

That is a healthy separation of concerns.


In [ ]:
@dataclass
class ServiceAccount:
    id: int
    name: str
    enabled: bool
    api_token: str
    internal_retry_count: int


### Step 2 — Create an explicit serializer

We do not serialize `__dict__`.

Instead, we deliberately allowlist the three public fields.


In [ ]:
class ServiceAccountSerializer(serpy.Serializer):
    id = serpy.IntField()
    name = serpy.StrField()
    enabled = serpy.BoolField()


### Step 3 — Serialize one instance


In [ ]:
account = ServiceAccount(
    id=17,
    name="build-bot",
    enabled=True,
    api_token="super-secret-token",
    internal_retry_count=4,
)

account_payload = ServiceAccountSerializer(account).data
pretty(account_payload)


### Step 4 — Verify the contract

Assertions are useful because serializers often become API contracts.

A regression that unexpectedly exposes a field can be more serious than a cosmetic change.


In [ ]:
assert account_payload == {
    "id": 17,
    "name": "build-bot",
    "enabled": True,
}

assert "api_token" not in account_payload
assert "internal_retry_count" not in account_payload


### Takeaway

A serializer is safer when it is an **allowlist**.

Avoid the pattern:

```python
payload = obj.__dict__.copy()
payload.pop("api_token")
```

That is a blocklist. A new sensitive attribute could later be added to the model and accidentally leak.

An explicit Serpy serializer does not include undeclared attributes.


# Part 2 — Decouple internal names from API names

Real systems accumulate legacy naming.

Your database model might use one name while the public API uses another.

Renaming the domain model just to make a serializer pretty can be unnecessarily invasive.

Serpy gives us two useful tools:

- `attr=` chooses where the value comes from;
- `label=` chooses what key appears in the output.


## Problem 2 — Map a legacy object to a modern API contract

Internal object:

```text
usr_id
profile.full_name
acct_state
```

Required output:

```json
{
  "userId": 91,
  "displayName": "Maya Chen",
  "status": "active"
}
```


### Step 1 — Build the internal model


In [ ]:
@dataclass
class LegacyProfile:
    full_name: str


@dataclass
class LegacyUser:
    usr_id: int
    profile: LegacyProfile
    acct_state: str


### Step 2 — Map source attributes

`attr="profile.full_name"` uses dotted attribute lookup.

That means we do not need to add a new `display_name` property merely for serialization.


In [ ]:
class LegacyUserSerializer(serpy.Serializer):
    user_id = serpy.IntField(
        attr="usr_id",
        label="userId",
    )

    display_name = serpy.StrField(
        attr="profile.full_name",
        label="displayName",
    )

    status = serpy.StrField(
        attr="acct_state",
    )


### Step 3 — Inspect the result


In [ ]:
legacy_user = LegacyUser(
    usr_id=91,
    profile=LegacyProfile("Maya Chen"),
    acct_state="active",
)

legacy_payload = LegacyUserSerializer(legacy_user).data
pretty(legacy_payload)


In [ ]:
assert legacy_payload == {
    "userId": 91,
    "displayName": "Maya Chen",
    "status": "active",
}


### Why both `attr` and `label`?

Consider this field:

```python
user_id = serpy.IntField(attr="usr_id", label="userId")
```

There are three names in play:

- `user_id` — Python name inside the serializer class;
- `usr_id` — source attribute on the object;
- `userId` — emitted API key.

Keeping these roles distinct makes migrations easier.


# Part 3 — Choose between `call=True` and `MethodField`

Both can produce derived output, but they solve different problems.

Use `call=True` when:

- the object already owns a useful zero-argument method;
- the serializer only needs to call it.

Use `MethodField` when:

- serialization logic combines several attributes;
- output formatting belongs in the serializer;
- you do not want to add presentation-only methods to the model.


## Problem 3 — Use the smallest appropriate mechanism

A `BuildJob` has:

- a zero-argument `short_id()` method;
- `completed_steps` and `total_steps`.

Output should include:

- the short ID by calling the object method;
- completion percentage by computing it in the serializer.


### Step 1 — Define the model


In [ ]:
@dataclass
class BuildJob:
    build_id: str
    completed_steps: int
    total_steps: int

    def short_id(self) -> str:
        return self.build_id[:8]


### Step 2 — Serialize the object method with `call=True`

The field name `short_id` matches the method name, so no `attr=` is necessary.


In [ ]:
class BuildJobSerializerV1(serpy.Serializer):
    short_id = serpy.StrField(call=True)


In [ ]:
job = BuildJob(
    build_id="6f90eab438c14d9a",
    completed_steps=7,
    total_steps=10,
)

print(BuildJobSerializerV1(job).data)


### Step 3 — Add a calculated field

Progress depends on two attributes.

That is a good `MethodField` use case.


In [ ]:
class BuildJobSerializer(serpy.Serializer):
    short_id = serpy.StrField(call=True)
    progress_percent = serpy.MethodField()

    def get_progress_percent(self, obj: BuildJob) -> float:
        if obj.total_steps == 0:
            return 0.0
        return round(
            obj.completed_steps / obj.total_steps * 100,
            1,
        )


In [ ]:
job_payload = BuildJobSerializer(job).data
pretty(job_payload)

assert job_payload == {
    "short_id": "6f90eab4",
    "progress_percent": 70.0,
}


### Design note

We could put `progress_percent()` on the model and use `call=True`.

That is not automatically wrong.

The question is architectural:

- Is progress a meaningful domain behavior?
- Or is it merely a presentation choice for one API?

If it is API-specific presentation logic, keeping it in the serializer is often cleaner.


# Part 4 — Build nested serializers one layer at a time

Nested serializers are easier to reason about if we avoid writing the entire graph at once.

We will start with the smallest child object and work outward.


## Problem 4 — Serialize a deployment with nested environment and owner objects

Required shape:

```json
{
  "deployment_id": 301,
  "environment": {
    "name": "production",
    "region": "eu-central"
  },
  "owner": {
    "id": 8,
    "name": "Release Engineering"
  }
}
```


### Step 1 — Define the models


In [ ]:
@dataclass
class Environment:
    name: str
    region: str
    internal_cluster_name: str


@dataclass
class Team:
    id: int
    name: str
    pager_rotation: str


@dataclass
class Deployment:
    deployment_id: int
    environment: Environment
    owner: Team


### Step 2 — Solve the environment independently

A nested serializer is still just a serializer.

We can test it before connecting it to a parent serializer.


In [ ]:
class EnvironmentSerializer(serpy.Serializer):
    name = serpy.StrField()
    region = serpy.StrField()


In [ ]:
prod = Environment(
    name="production",
    region="eu-central",
    internal_cluster_name="prod-k8s-04",
)

environment_payload = EnvironmentSerializer(prod).data
print(environment_payload)

assert environment_payload == {
    "name": "production",
    "region": "eu-central",
}


### Step 3 — Solve the owner independently


In [ ]:
class TeamSummarySerializer(serpy.Serializer):
    id = serpy.IntField()
    name = serpy.StrField()


In [ ]:
release_team = Team(
    id=8,
    name="Release Engineering",
    pager_rotation="release-primary",
)

assert TeamSummarySerializer(release_team).data == {
    "id": 8,
    "name": "Release Engineering",
}


### Step 4 — Compose the parent serializer

A `Serializer` is also usable as a field.

This is the mechanism behind nested schemas.


In [ ]:
class DeploymentSerializer(serpy.Serializer):
    deployment_id = serpy.IntField()
    environment = EnvironmentSerializer()
    owner = TeamSummarySerializer()


In [ ]:
deployment = Deployment(
    deployment_id=301,
    environment=prod,
    owner=release_team,
)

deployment_payload = DeploymentSerializer(deployment).data
pretty(deployment_payload)


In [ ]:
assert deployment_payload == {
    "deployment_id": 301,
    "environment": {
        "name": "production",
        "region": "eu-central",
    },
    "owner": {
        "id": 8,
        "name": "Release Engineering",
    },
}


### Takeaway

Test nested serializers from the inside out.

When a large payload is wrong, this gives you clear fault boundaries:

- child serializer incorrect?
- parent mapping incorrect?
- source object incorrect?

That is much easier to debug than one giant serializer written in a single step.


# Part 5 — Serialize collections with `many=True`

`many=True` changes the serializer from:

```text
one object -> one dictionary
```

to:

```text
iterable of objects -> list of dictionaries
```

The same idea also applies to nested serializers.


## Problem 5 — Serialize an incident timeline

An incident contains many timeline events.

We want:

```json
{
  "incident_id": "INC-204",
  "events": [
    {"sequence": 1, "message": "..."},
    {"sequence": 2, "message": "..."}
  ]
}
```


### Step 1 — Define event and incident models


In [ ]:
@dataclass
class IncidentEvent:
    sequence: int
    message: str
    internal_debug_context: str


@dataclass
class Incident:
    incident_id: str
    events: list[IncidentEvent]


### Step 2 — Serialize a single event first


In [ ]:
class IncidentEventSerializer(serpy.Serializer):
    sequence = serpy.IntField()
    message = serpy.StrField()


In [ ]:
event_1 = IncidentEvent(
    sequence=1,
    message="Alert triggered",
    internal_debug_context="monitor=latency-p99",
)

assert IncidentEventSerializer(event_1).data == {
    "sequence": 1,
    "message": "Alert triggered",
}


### Step 3 — Nest the event serializer with `many=True`


In [ ]:
class IncidentSerializer(serpy.Serializer):
    incident_id = serpy.StrField()
    events = IncidentEventSerializer(many=True)


In [ ]:
incident = Incident(
    incident_id="INC-204",
    events=[
        event_1,
        IncidentEvent(
            sequence=2,
            message="On-call acknowledged",
            internal_debug_context="user=42",
        ),
        IncidentEvent(
            sequence=3,
            message="Mitigation deployed",
            internal_debug_context="deploy=301",
        ),
    ],
)

incident_payload = IncidentSerializer(incident).data
pretty(incident_payload)

assert len(incident_payload["events"]) == 3
assert incident_payload["events"][2]["sequence"] == 3


### Common mistake

This is wrong for a list:

```python
events = IncidentEventSerializer()
```

That tells the nested serializer to treat the entire list as if it were one event object.

The collection boundary must be explicit:

```python
events = IncidentEventSerializer(many=True)
```


# Part 6 — Optional values: distinguish `None` from missing attributes

Optional output is more subtle than it first appears.

With `required=False`, Serpy can tolerate:

- a missing attribute/key;
- a value of `None`.

But those two cases do not necessarily produce identical output.


## Problem 6 — Observe the three-state behavior

We will serialize a `description` field for three objects:

1. `"ready"` — attribute exists with a value;
2. `None` — attribute exists but is empty;
3. attribute does not exist.


### Step 1 — Create objects with deliberately different shapes


In [ ]:
class FlexibleTask:
    def __init__(self, task_id, description_marker=...):
        self.task_id = task_id

        if description_marker is not ...:
            self.description = description_marker


### Step 2 — Mark the field `required=False`


In [ ]:
class FlexibleTaskSerializer(serpy.Serializer):
    task_id = serpy.IntField()
    description = serpy.StrField(required=False)


### Step 3 — Serialize all three cases together


In [ ]:
tasks = [
    FlexibleTask(1, "ready"),
    FlexibleTask(2, None),
    FlexibleTask(3),
]

task_payloads = FlexibleTaskSerializer(tasks, many=True).data
pretty(task_payloads)


### Step 4 — State the expected semantics explicitly


In [ ]:
assert task_payloads == [
    {"task_id": 1, "description": "ready"},
    {"task_id": 2, "description": None},
    {"task_id": 3},
]


### Why this matters

Some APIs distinguish:

```json
{"description": null}
```

from:

```json
{}
```

`null` can mean "known to be empty."

Missing can mean "not requested", "not loaded", "not applicable", or "not present."

Do not collapse these states accidentally if your API contract cares about them.


# Part 7 — Build reusable scalar conversions with custom fields

If the same conversion appears repeatedly, copying `MethodField` methods into many serializers creates duplication.

A custom Serpy field can turn the conversion into a reusable primitive.


## Problem 7 — Serialize timezone-aware timestamps consistently

Requirements:

- input must be a `datetime`;
- timestamp must be timezone-aware;
- output should be ISO 8601;
- timestamps should be normalized to UTC;
- invalid values should fail loudly.


### Step 1 — Decide where the conversion belongs

This is a scalar transformation:

```text
datetime -> string
```

That is an excellent `Field.to_value()` use case.


In [ ]:
class UTCDateTimeField(serpy.Field):
    def to_value(self, value: datetime) -> str:
        if not isinstance(value, datetime):
            raise TypeError(
                f"expected datetime, got {type(value).__name__}"
            )

        if value.tzinfo is None:
            raise ValueError(
                "datetime must be timezone-aware"
            )

        return value.astimezone(timezone.utc).isoformat()


### Step 2 — Use the field in a serializer


In [ ]:
@dataclass
class AuditEntry:
    action: str
    happened_at: datetime


class AuditEntrySerializer(serpy.Serializer):
    action = serpy.StrField()
    happened_at = UTCDateTimeField()


### Step 3 — Verify a valid timestamp


In [ ]:
audit = AuditEntry(
    action="deployment.completed",
    happened_at=datetime(
        2026, 8, 7, 15, 45,
        tzinfo=timezone.utc,
    ),
)

audit_payload = AuditEntrySerializer(audit).data
print(audit_payload)

assert audit_payload == {
    "action": "deployment.completed",
    "happened_at": "2026-08-07T15:45:00+00:00",
}


### Step 4 — Verify an invalid timestamp

Serialization is often the last boundary before data leaves your application.

Failing loudly can be preferable to silently publishing ambiguous time data.


In [ ]:
try:
    AuditEntrySerializer(
        AuditEntry(
            action="bad.timestamp",
            happened_at=datetime(2026, 8, 7, 15, 45),
        )
    ).data
except ValueError as exc:
    print("Expected error:", exc)
else:
    raise AssertionError("Expected timezone validation failure")


### Important distinction

This custom field performs **output-side sanity checking**.

That does not make Serpy an inbound validation framework.

Invalid external input should still be validated before domain objects are created.


# Part 8 — Handle money deliberately

Monetary values are a classic serialization trap.

Consider:

```python
Decimal("10.10")
```

Converting it to `float` changes the numeric representation model.

Many APIs instead choose a decimal string at the serialization boundary.


## Problem 8 — Create a currency field and compute billing totals

We want exact two-decimal strings:

```json
{
  "unit_price": "19.99",
  "subtotal": "59.97"
}
```


### Step 1 — Make one reusable money field


In [ ]:
class MoneyField(serpy.Field):
    def to_value(self, value: Any) -> str:
        amount = (
            value
            if isinstance(value, Decimal)
            else Decimal(str(value))
        )

        rounded = amount.quantize(
            Decimal("0.01"),
            rounding=ROUND_HALF_UP,
        )

        return format(rounded, ".2f")


### Step 2 — Define the billing model


In [ ]:
@dataclass
class BillingLine:
    description: str
    quantity: int
    unit_price: Decimal


### Step 3 — Use the custom field for direct values

`unit_price` is already on the object, so `MoneyField` is enough.


In [ ]:
class BillingLineSerializerV1(serpy.Serializer):
    description = serpy.StrField()
    quantity = serpy.IntField()
    unit_price = MoneyField()


In [ ]:
billing_line = BillingLine(
    description="Priority support hour",
    quantity=3,
    unit_price=Decimal("19.99"),
)

pretty(BillingLineSerializerV1(billing_line).data)


### Step 4 — Use `MethodField` for the computed subtotal

The subtotal depends on quantity *and* price.

We can still return the same string representation used by `MoneyField`.


In [ ]:
class BillingLineSerializer(serpy.Serializer):
    description = serpy.StrField()
    quantity = serpy.IntField()
    unit_price = MoneyField()
    subtotal = serpy.MethodField()

    def get_subtotal(self, obj: BillingLine) -> str:
        value = obj.unit_price * obj.quantity
        rounded = value.quantize(
            Decimal("0.01"),
            rounding=ROUND_HALF_UP,
        )
        return format(rounded, ".2f")


In [ ]:
billing_payload = BillingLineSerializer(billing_line).data
pretty(billing_payload)

assert billing_payload == {
    "description": "Priority support hour",
    "quantity": 3,
    "unit_price": "19.99",
    "subtotal": "59.97",
}


### Design improvement

In a larger codebase, we would probably factor the two-decimal formatting into one shared function so the custom field and computed methods cannot drift apart.

Serialization code is still production code: duplication can create inconsistent API behavior.


# Part 9 — Go beyond `to_value()` with a custom getter

`to_value()` answers:

> "How should I transform the value after Serpy fetches it?"

`as_getter()` answers a deeper question:

> "How should Serpy obtain the value in the first place?"

This is useful when normal attribute access is not enough.


## Problem 9 — Build a serializer-aware prefix field

We want to reuse the same serializer class with different external ID prefixes.

Example:

```text
raw id: 312
prefix: "usr_"
output: "usr_312"
```

The prefix belongs to the serializer instance, not the model.


### Step 1 — Understand `getter_takes_serializer`

Normally, a field getter receives only the object being serialized.

A custom field can set:

```python
getter_takes_serializer = True
```

Then the getter receives:

```text
serializer instance + source object
```

This is an advanced extension point.


In [ ]:
class PrefixedField(serpy.Field):
    getter_takes_serializer = True

    def as_getter(
        self,
        serializer_field_name,
        serializer_cls,
    ):
        source_name = self.attr or serializer_field_name
        source_getter = operator.attrgetter(source_name)

        def get_prefixed(serializer, obj):
            raw_value = source_getter(obj)
            return f"{serializer.id_prefix}{raw_value}"

        return get_prefixed


### Step 2 — Give the serializer instance configuration

Serpy's standard `context` parameter is documented as compatibility-oriented and unused, so here we make the configuration explicit with our own constructor argument.


In [ ]:
@dataclass
class ExternalUser:
    id: int
    name: str


class ExternalUserSerializer(serpy.Serializer):
    external_id = PrefixedField(attr="id")
    name = serpy.StrField()

    def __init__(
        self,
        instance=None,
        many=False,
        id_prefix="usr_",
        **kwargs,
    ):
        self.id_prefix = id_prefix
        super().__init__(
            instance=instance,
            many=many,
            **kwargs,
        )


### Step 3 — Reuse the same schema with different prefixes


In [ ]:
external_user = ExternalUser(312, "Nora")

public_payload = ExternalUserSerializer(
    external_user,
    id_prefix="usr_",
).data

legacy_payload = ExternalUserSerializer(
    external_user,
    id_prefix="legacy-user-",
).data

print(public_payload)
print(legacy_payload)

assert public_payload["external_id"] == "usr_312"
assert legacy_payload["external_id"] == "legacy-user-312"


### When *not* to do this

`as_getter()` is powerful, but it is more complex than ordinary fields.

Prefer, in order:

1. normal field;
2. `attr=`;
3. `call=True`;
4. `MethodField`;
5. custom `to_value()`;
6. custom `as_getter()`.

Use the more advanced mechanism only when the simpler one cannot express the requirement cleanly.


# Part 10 — Serialize dictionaries with `DictSerializer`

Not all application boundaries hand you rich objects.

Sometimes a database adapter, aggregation layer, or cache already returns dictionaries.

`DictSerializer` uses key lookup rather than attribute lookup.


## Problem 10 — Normalize analytics rows

Input rows contain numeric strings:

```python
{
    "endpoint": "/search",
    "request_count": "4312",
    "error_rate": "0.012"
}
```

We want typed native output.


### Step 1 — Define a dictionary serializer


In [ ]:
class EndpointMetricSerializer(serpy.DictSerializer):
    endpoint = serpy.StrField()
    request_count = serpy.IntField()
    error_rate = serpy.FloatField()


### Step 2 — Serialize multiple rows

Because the source is a list of dictionaries, use `many=True`.


In [ ]:
metric_rows = [
    {
        "endpoint": "/search",
        "request_count": "4312",
        "error_rate": "0.012",
    },
    {
        "endpoint": "/checkout",
        "request_count": "987",
        "error_rate": "0.034",
    },
]

metric_payload = EndpointMetricSerializer(
    metric_rows,
    many=True,
).data

pretty(metric_payload)


### Step 3 — Verify conversion, not only values

A string `"4312"` and an integer `4312` can look similar in printed output.

Type assertions help verify the serializer's boundary behavior.


In [ ]:
assert metric_payload[0] == {
    "endpoint": "/search",
    "request_count": 4312,
    "error_rate": 0.012,
}

assert isinstance(
    metric_payload[0]["request_count"],
    int,
)

assert isinstance(
    metric_payload[0]["error_rate"],
    float,
)


### Architectural note

Use `DictSerializer` when dictionary-shaped input is already the natural result of the layer before serialization.

Do not convert every domain object to a dictionary merely so you can use `DictSerializer`; normal `Serializer` already handles objects directly.


# Part 11 — Reuse schemas with inheritance and mixins

Serializer inheritance is useful when several API resources share a stable set of fields.

The danger is creating a huge inheritance hierarchy that hides what each API exposes.

We will use inheritance only for a small, obvious cross-cutting concern.


## Problem 11 — Add audit metadata consistently

Several public resources expose:

- `created_at`
- `updated_at`

We want one reusable serializer mixin.


### Step 1 — Define a reusable audit serializer


In [ ]:
class AuditFieldsSerializer(serpy.Serializer):
    created_at = UTCDateTimeField()
    updated_at = UTCDateTimeField()


### Step 2 — Define a resource model


In [ ]:
@dataclass
class FeatureFlag:
    key: str
    enabled: bool
    created_at: datetime
    updated_at: datetime
    internal_rule_expression: str


### Step 3 — Inherit the common output fields


In [ ]:
class FeatureFlagSerializer(AuditFieldsSerializer):
    key = serpy.StrField()
    enabled = serpy.BoolField()


In [ ]:
flag = FeatureFlag(
    key="new-checkout",
    enabled=True,
    created_at=datetime(
        2026, 8, 1, 10, 0,
        tzinfo=timezone.utc,
    ),
    updated_at=datetime(
        2026, 8, 7, 17, 15,
        tzinfo=timezone.utc,
    ),
    internal_rule_expression="country in ('BG','DE')",
)

flag_payload = FeatureFlagSerializer(flag).data
pretty(flag_payload)

assert set(flag_payload) == {
    "key",
    "enabled",
    "created_at",
    "updated_at",
}


### Best-practice rule

Inheritance should make the public schema **easier** to understand.

If discovering an API response requires mentally expanding five base classes, composition or a small amount of duplication may be clearer.


# Part 12 — Design finite object graphs

Domain models often contain back-references:

```text
Project -> tasks -> project -> tasks -> project -> ...
```

A serializer should not blindly mirror that graph.

API payloads need intentional boundaries.


## Problem 12 — Break a circular project/task relationship

Domain relationship:

- a project contains tasks;
- each task references its project.

Public design:

- project output includes task summaries;
- task output includes only `project_id`.

That creates a finite graph.


### Step 1 — Model the circular relationship


In [ ]:
@dataclass
class Project:
    id: int
    name: str
    tasks: list["ProjectTask"] = field(default_factory=list)


@dataclass
class ProjectTask:
    id: int
    title: str
    project: Project | None = None


### Step 2 — Create a task summary

The task summary deliberately does **not** nest the project.


In [ ]:
class ProjectTaskSummarySerializer(serpy.Serializer):
    id = serpy.IntField()
    title = serpy.StrField()


### Step 3 — Create the project serializer


In [ ]:
class ProjectSerializer(serpy.Serializer):
    id = serpy.IntField()
    name = serpy.StrField()
    tasks = ProjectTaskSummarySerializer(many=True)


### Step 4 — Create a standalone task serializer

For a task endpoint, we still want a link to the parent.

A scalar ID is enough.


In [ ]:
class ProjectTaskSerializer(serpy.Serializer):
    id = serpy.IntField()
    title = serpy.StrField()
    project_id = serpy.IntField(attr="project.id")


### Step 5 — Build and serialize the graph


In [ ]:
project = Project(44, "Serializer Migration")

task_a = ProjectTask(
    1,
    "Audit current payloads",
    project,
)
task_b = ProjectTask(
    2,
    "Add contract tests",
    project,
)

project.tasks.extend([task_a, task_b])

project_payload = ProjectSerializer(project).data
task_payload = ProjectTaskSerializer(task_a).data

pretty(project_payload)
pretty(task_payload)

assert task_payload["project_id"] == 44
assert "project" not in project_payload["tasks"][0]


### Design lesson

Good serialization is not graph traversal.

It is **representation design**.

Ask what the consumer needs, then stop the graph at a sensible boundary.


# Part 13 — Version an API without changing the domain model

Serializers can serve as an anti-corruption layer between internal object structure and external contracts.

This is especially useful during API migrations.


## Problem 13 — Maintain v1 while introducing v2

Domain object:

```text
name
plan_code
seat_count
```

V1 response:

```json
{
  "name": "...",
  "plan": "pro"
}
```

V2 response additionally contains:

```json
{
  "seats": 25,
  "tierLabel": "PRO"
}
```


### Step 1 — Define the domain model once


In [ ]:
@dataclass
class Subscription:
    name: str
    plan_code: str
    seat_count: int


### Step 2 — Preserve the existing v1 schema


In [ ]:
class SubscriptionV1Serializer(serpy.Serializer):
    name = serpy.StrField()
    plan = serpy.StrField(attr="plan_code")


### Step 3 — Extend for v2

V2 inherits the stable v1 fields and adds new representation choices.


In [ ]:
class SubscriptionV2Serializer(
    SubscriptionV1Serializer
):
    seats = serpy.IntField(attr="seat_count")
    tier_label = serpy.MethodField(
        label="tierLabel"
    )

    def get_tier_label(
        self,
        obj: Subscription,
    ) -> str:
        return obj.plan_code.upper()


### Step 4 — Compare contracts side by side


In [ ]:
subscription = Subscription(
    name="Example Workspace",
    plan_code="pro",
    seat_count=25,
)

v1_payload = SubscriptionV1Serializer(
    subscription
).data

v2_payload = SubscriptionV2Serializer(
    subscription
).data

print("v1")
pretty(v1_payload)

print("v2")
pretty(v2_payload)


In [ ]:
assert v1_payload == {
    "name": "Example Workspace",
    "plan": "pro",
}

assert v2_payload == {
    "name": "Example Workspace",
    "plan": "pro",
    "seats": 25,
    "tierLabel": "PRO",
}


### Why this is useful

The domain object did not acquire:

- `tierLabel`;
- `plan`;
- version-specific properties.

Those are representation concerns.

The serializer absorbs the contract differences.


# Part 14 — Treat conversion failures as boundary signals

Serpy's typed fields call Python conversions such as `int(...)` and `float(...)`.

That can fail.

The important question is not "Can Serpy throw an error?"

The important question is "At which layer should invalid data normally be caught?"


## Problem 14 — Explore the difference between serialization failure and input validation

We will intentionally serialize an invalid age.


### Step 1 — Define a simple serializer


In [ ]:
@dataclass
class OperatorProfile:
    name: str
    years_experience: Any


class OperatorProfileSerializer(serpy.Serializer):
    name = serpy.StrField()
    years_experience = serpy.IntField()


### Step 2 — Observe a successful conversion

A numeric string is compatible with `int(...)`.


In [ ]:
compatible_profile = OperatorProfile(
    "Alex",
    "12",
)

compatible_payload = OperatorProfileSerializer(
    compatible_profile
).data

print(compatible_payload)

assert compatible_payload["years_experience"] == 12
assert isinstance(
    compatible_payload["years_experience"],
    int,
)


### Step 3 — Observe an incompatible value


In [ ]:
incompatible_profile = OperatorProfile(
    "Taylor",
    "twelve",
)

try:
    OperatorProfileSerializer(
        incompatible_profile
    ).data
except ValueError as exc:
    print("Expected conversion failure:", exc)
else:
    raise AssertionError("Expected ValueError")


### Step 4 — Try to use Serpy as an input validator

This is intentionally incorrect.


In [ ]:
try:
    OperatorProfileSerializer(
        data={
            "name": "Jordan",
            "years_experience": 7,
        }
    )
except RuntimeError as exc:
    print("Expected Serpy limitation:", exc)
else:
    raise AssertionError(
        "Expected Serpy to reject data=..."
    )


### Architecture takeaway

A reasonable pipeline is:

```text
untrusted request
    |
    v
validation / parsing layer
    |
    v
trusted domain object
    |
    v
Serpy output serializer
    |
    v
response payload
```

A conversion exception during serialization may reveal a broken invariant, but it is not a substitute for friendly validation of user input.


# Part 15 — Understand `.data` caching

Serpy caches the result of `.data` on a serializer instance.

This is normally convenient.

It can become surprising if you mutate the source object and reuse the same serializer instance.


## Problem 15 — Demonstrate stale serialized state

We will:

1. create an object;
2. create one serializer instance;
3. access `.data`;
4. mutate the object;
5. access `.data` again;
6. compare with a new serializer instance.


In [ ]:
@dataclass
class LiveCounter:
    name: str
    value: int


class LiveCounterSerializer(serpy.Serializer):
    name = serpy.StrField()
    value = serpy.IntField()


### Step 1 — First serialization


In [ ]:
counter = LiveCounter("workers", 3)
counter_serializer = LiveCounterSerializer(counter)

first_payload = counter_serializer.data
print("first:", first_payload)


### Step 2 — Mutate the source object


In [ ]:
counter.value = 8
print("object now:", counter)


### Step 3 — Reuse the serializer instance


In [ ]:
second_payload = counter_serializer.data
print("same serializer:", second_payload)

assert second_payload["value"] == 3


### Step 4 — Create a fresh serializer


In [ ]:
fresh_payload = LiveCounterSerializer(counter).data
print("fresh serializer:", fresh_payload)

assert fresh_payload["value"] == 8


### Best practice

Treat serializer instances as short-lived renderers.

A simple style is:

```python
payload = SomeSerializer(obj).data
```

rather than storing serializer instances and reusing them after model mutation.


# Part 16 — Keep Serpy separate from JSON and YAML encoding

Serpy normally produces native Python structures.

Those structures can then be encoded by a boundary-specific library.

This separation is worth preserving.


## Problem 16 — Build three layers of one response

We will distinguish:

1. domain object;
2. Serpy native payload;
3. JSON/YAML representation.

This sounds simple, but explicitly separating the layers makes testing and performance analysis much clearer.


### Step 1 — Serialize to native Python data


In [ ]:
native_payload = DeploymentSerializer(
    deployment
).data

print(type(native_payload))
print(type(native_payload["environment"]))


At this point there is no JSON string.

We have ordinary Python dictionaries and scalar values.


### Step 2 — Encode JSON


In [ ]:
json_text = json.dumps(
    native_payload,
    indent=2,
    sort_keys=True,
)

print(json_text)

assert isinstance(json_text, str)


### Step 3 — Encode safe YAML

YAML is a separate representation of the same native payload.


In [ ]:
yaml_text = yaml.safe_dump(
    native_payload,
    sort_keys=False,
)

print(yaml_text)

assert isinstance(yaml_text, str)


### Step 4 — Verify JSON round-trip equivalence


In [ ]:
decoded_again = json.loads(json_text)

assert decoded_again == native_payload


### Why this layering matters

Suppose response time is slow.

If the stages are separate, you can measure:

- model/database time;
- Serpy time;
- JSON encoding time;
- network time.

"Serialization is slow" is otherwise too vague to optimize responsibly.


# Part 17 — Benchmark a realistic shape instead of repeating marketing numbers

Library benchmarks are useful for orientation.

Your workload is what matters.

We will benchmark:

- Serpy object-to-native conversion;
- JSON encoding of the already-serialized native data;
- the complete Serpy + JSON path.

We are not trying to prove a universal ranking.


## Problem 17 — Measure 5,000 incident events

First create production-shaped test data.


In [ ]:
benchmark_events = [
    IncidentEvent(
        sequence=i,
        message=f"event-{i}",
        internal_debug_context=f"trace-{i}",
    )
    for i in range(5_000)
]


### Step 1 — Define the operations

Notice that `json_only()` reuses an already-built native payload.

That isolates JSON encoding from Serpy's work.


In [ ]:
benchmark_native = IncidentEventSerializer(
    benchmark_events,
    many=True,
).data


def serpy_only():
    return IncidentEventSerializer(
        benchmark_events,
        many=True,
    ).data


def json_only():
    return json.dumps(benchmark_native)


def serpy_and_json():
    return json.dumps(
        IncidentEventSerializer(
            benchmark_events,
            many=True,
        ).data
    )


### Step 2 — Verify correctness before timing

Never benchmark an implementation that is not known to produce the correct shape.


In [ ]:
assert len(serpy_only()) == 5_000
assert len(json.loads(json_only())) == 5_000
assert len(json.loads(serpy_and_json())) == 5_000


### Step 3 — Run repeated timings

We use the best time from several repeats as a simple way to reduce one-off environmental noise.

For serious benchmarking, use a dedicated benchmarking tool and a controlled environment.


In [ ]:
serpy_times = timeit.repeat(
    serpy_only,
    number=5,
    repeat=5,
)

json_times = timeit.repeat(
    json_only,
    number=5,
    repeat=5,
)

combined_times = timeit.repeat(
    serpy_and_json,
    number=5,
    repeat=5,
)

print(
    "Serpy only:    ",
    f"{min(serpy_times):.6f}s",
)
print(
    "JSON only:     ",
    f"{min(json_times):.6f}s",
)
print(
    "Serpy + JSON:  ",
    f"{min(combined_times):.6f}s",
)


### Interpretation questions

When you run the benchmark, ask:

- Which stage is largest?
- Does JSON encoding dominate?
- Does a more deeply nested object shape change the balance?
- What happens if computed `MethodField`s become expensive?
- What happens if a field calls a database-backed property?

A serializer library cannot compensate for expensive getters or accidental I/O inside the object graph.


# Part 18 — Capstone: design a release-report API step by step

The capstone combines many earlier ideas.

We will intentionally **not** write the final serializer first.

We will build the representation from small pieces, test each piece, then compose them.


## Problem 18 — Release report

A release report contains:

- release metadata;
- an owner team;
- a list of deployed components;
- a list of checks;
- internal notes that must never be exposed.

We want the final public shape:

```json
{
  "releaseId": "rel_2026_08_07",
  "version": "4.8.0",
  "releasedAt": "...",
  "owner": {
    "id": 8,
    "name": "Release Engineering"
  },
  "components": [
    {
      "name": "api",
      "previous": "4.7.2",
      "current": "4.8.0",
      "changed": true
    }
  ],
  "checks": [
    {
      "name": "smoke-tests",
      "passed": true
    }
  ],
  "summary": {
    "componentCount": 1,
    "changedCount": 1,
    "allChecksPassed": true
  }
}
```


### Step 1 — Model the domain

The model contains more information than the API needs.

That is normal.


In [ ]:
@dataclass
class ReleaseComponent:
    name: str
    previous_version: str
    current_version: str
    internal_artifact_digest: str


@dataclass
class ReleaseCheck:
    name: str
    passed: bool
    diagnostic_details: str


@dataclass
class ReleaseReport:
    release_id: str
    version: str
    released_at: datetime
    owner: Team
    components: list[ReleaseComponent]
    checks: list[ReleaseCheck]
    internal_notes: str


### Step 2 — Solve one component

Required public component fields:

- name
- previous
- current
- changed

The digest is internal and omitted.


In [ ]:
class ReleaseComponentSerializer(serpy.Serializer):
    name = serpy.StrField()

    previous = serpy.StrField(
        attr="previous_version"
    )

    current = serpy.StrField(
        attr="current_version"
    )

    changed = serpy.MethodField()

    def get_changed(
        self,
        obj: ReleaseComponent,
    ) -> bool:
        return (
            obj.previous_version
            != obj.current_version
        )


### Step 3 — Test the component in isolation


In [ ]:
api_component = ReleaseComponent(
    name="api",
    previous_version="4.7.2",
    current_version="4.8.0",
    internal_artifact_digest="sha256:secret-ish-internal-value",
)

component_payload = ReleaseComponentSerializer(
    api_component
).data

pretty(component_payload)

assert component_payload == {
    "name": "api",
    "previous": "4.7.2",
    "current": "4.8.0",
    "changed": True,
}


### Step 4 — Solve one check

Diagnostic details are intentionally private.


In [ ]:
class ReleaseCheckSerializer(serpy.Serializer):
    name = serpy.StrField()
    passed = serpy.BoolField()


In [ ]:
smoke_check = ReleaseCheck(
    name="smoke-tests",
    passed=True,
    diagnostic_details="internal log pointer",
)

assert ReleaseCheckSerializer(
    smoke_check
).data == {
    "name": "smoke-tests",
    "passed": True,
}


### Step 5 — Decide how to represent summary data

The summary does not exist as a nested object on the model.

It is a presentation calculated from the report.

A `MethodField` can return an ordinary dictionary.

This keeps the model free from an API-only `summary` object.


### Step 6 — Assemble the top-level serializer


In [ ]:
class ReleaseReportSerializer(serpy.Serializer):
    release_id = serpy.StrField(
        label="releaseId"
    )

    version = serpy.StrField()

    released_at = UTCDateTimeField(
        label="releasedAt"
    )

    owner = TeamSummarySerializer()

    components = ReleaseComponentSerializer(
        many=True
    )

    checks = ReleaseCheckSerializer(
        many=True
    )

    summary = serpy.MethodField()

    def get_summary(
        self,
        obj: ReleaseReport,
    ) -> dict[str, Any]:
        changed_count = sum(
            component.previous_version
            != component.current_version
            for component in obj.components
        )

        return {
            "componentCount": len(
                obj.components
            ),
            "changedCount": changed_count,
            "allChecksPassed": all(
                check.passed
                for check in obj.checks
            ),
        }


### Step 7 — Create realistic data


In [ ]:
release_report = ReleaseReport(
    release_id="rel_2026_08_07",
    version="4.8.0",
    released_at=datetime(
        2026, 8, 7, 18, 30,
        tzinfo=timezone.utc,
    ),
    owner=release_team,
    components=[
        api_component,
        ReleaseComponent(
            name="worker",
            previous_version="4.8.0",
            current_version="4.8.0",
            internal_artifact_digest="sha256:worker",
        ),
        ReleaseComponent(
            name="web",
            previous_version="4.7.9",
            current_version="4.8.0",
            internal_artifact_digest="sha256:web",
        ),
    ],
    checks=[
        smoke_check,
        ReleaseCheck(
            name="database-migrations",
            passed=True,
            diagnostic_details="migration set 47",
        ),
        ReleaseCheck(
            name="synthetic-checkout",
            passed=True,
            diagnostic_details="trace 112233",
        ),
    ],
    internal_notes=(
        "Do not publish these operator notes."
    ),
)


### Step 8 — Serialize and inspect the whole report


In [ ]:
release_payload = ReleaseReportSerializer(
    release_report
).data

pretty(release_payload)


### Step 9 — Verify nested behavior


In [ ]:
assert release_payload["releaseId"] == (
    "rel_2026_08_07"
)

assert release_payload["releasedAt"] == (
    "2026-08-07T18:30:00+00:00"
)

assert release_payload["owner"] == {
    "id": 8,
    "name": "Release Engineering",
}

assert release_payload["components"][0] == {
    "name": "api",
    "previous": "4.7.2",
    "current": "4.8.0",
    "changed": True,
}

assert release_payload["summary"] == {
    "componentCount": 3,
    "changedCount": 2,
    "allChecksPassed": True,
}


### Step 10 — Verify secret/internal data cannot leak

Test the public contract explicitly.


In [ ]:
serialized_text = json.dumps(release_payload)

forbidden_fragments = [
    "internal_notes",
    "internal_artifact_digest",
    "diagnostic_details",
    "Do not publish these operator notes.",
    "sha256:",
    "trace 112233",
]

for fragment in forbidden_fragments:
    assert fragment not in serialized_text


### Step 11 — Put the resource inside an API envelope

Pagination, request IDs, and API metadata often belong outside the resource serializer.

This keeps the resource serializer focused on one domain representation.


In [ ]:
api_response = {
    "data": release_payload,
    "meta": {
        "schemaVersion": "2026-08",
        "requestId": "req_f3a9",
    },
}

pretty(api_response)


### Step 12 — Encode the final response

Serpy's work is complete before this line.

JSON encoding is a separate boundary operation.


In [ ]:
response_json = json.dumps(
    api_response,
    separators=(",", ":"),
)

assert json.loads(response_json) == api_response

print(response_json[:180] + "...")


### Capstone review

This final serializer uses:

- explicit public allowlists;
- `label=` for API naming;
- a custom datetime field;
- nested one-to-one serialization;
- nested one-to-many serialization;
- `MethodField` for derived booleans;
- `MethodField` for a derived nested dictionary;
- separation between Serpy and JSON;
- contract assertions;
- negative assertions for secret leakage.

More importantly, we built the payload **from the leaves inward**.

That is a reliable way to design complex serializers.


# Part 19 — Additional guided challenge: flatten nested infrastructure data

This problem reinforces a principle:

Do not use a `MethodField` when `attr=` can express the mapping directly.


## Problem 19 — Flatten a node's region and provider

Input:

```text
node.cloud.provider
node.cloud.region
node.hostname
```

Output:

```json
{
  "hostname": "node-7",
  "provider": "example-cloud",
  "region": "eu-1"
}
```


In [ ]:
@dataclass
class CloudLocation:
    provider: str
    region: str


@dataclass
class ComputeNode:
    hostname: str
    cloud: CloudLocation


### Solution reasoning

There is no calculation.

We only need dotted source lookup.

Therefore `attr=` is simpler than `MethodField`.


In [ ]:
class ComputeNodeSerializer(serpy.Serializer):
    hostname = serpy.StrField()
    provider = serpy.StrField(
        attr="cloud.provider"
    )
    region = serpy.StrField(
        attr="cloud.region"
    )


In [ ]:
node = ComputeNode(
    hostname="node-7",
    cloud=CloudLocation(
        provider="example-cloud",
        region="eu-1",
    ),
)

node_payload = ComputeNodeSerializer(node).data
pretty(node_payload)

assert node_payload == {
    "hostname": "node-7",
    "provider": "example-cloud",
    "region": "eu-1",
}


# Part 20 — Additional guided challenge: optional nested serializer

Optional nested objects combine two concepts:

- nested serializers;
- `required=False`.

We will verify both a real child object and `None`.


## Problem 20 — Optional escalation policy

A monitoring rule may have an escalation policy, or it may explicitly have none.


In [ ]:
@dataclass
class EscalationPolicy:
    id: int
    name: str


@dataclass
class MonitorRule:
    name: str
    escalation: EscalationPolicy | None


### Solution

A nested serializer is itself a field, so it can be marked `required=False`.


In [ ]:
class EscalationPolicySerializer(serpy.Serializer):
    id = serpy.IntField()
    name = serpy.StrField()


class MonitorRuleSerializer(serpy.Serializer):
    name = serpy.StrField()
    escalation = EscalationPolicySerializer(
        required=False
    )


In [ ]:
rules = [
    MonitorRule(
        "checkout-errors",
        EscalationPolicy(
            5,
            "Commerce On-call",
        ),
    ),
    MonitorRule(
        "nightly-cleanup",
        None,
    ),
]

rule_payloads = MonitorRuleSerializer(
    rules,
    many=True,
).data

pretty(rule_payloads)

assert rule_payloads == [
    {
        "name": "checkout-errors",
        "escalation": {
            "id": 5,
            "name": "Commerce On-call",
        },
    },
    {
        "name": "nightly-cleanup",
        "escalation": None,
    },
]


### Takeaway

`required=False` does not mean "drop every `None` field."

Here the attribute exists and is `None`, so `None` remains visible in the output.

Missing and explicitly null remain different states.


# Final review — Decision guide

When designing a Serpy field, ask these questions in order.

### 1. Is the source value already on the object?

Use a normal field:

```python
name = serpy.StrField()
```

### 2. Is the value on a differently named or nested attribute?

Use `attr=`:

```python
region = serpy.StrField(attr="cloud.region")
```

### 3. Does the object already have a useful zero-argument method?

Use `call=True`:

```python
short_id = serpy.StrField(call=True)
```

### 4. Does the output depend on several attributes?

Use `MethodField`:

```python
total = serpy.MethodField()
```

### 5. Is the same scalar conversion repeated?

Create a custom field and override `to_value()`.

### 6. Does normal attribute/key retrieval itself need to change?

Only then consider `as_getter()`.


# Final review — Architecture checklist

Before shipping a serializer, ask:

- Does it expose only intentionally public fields?
- Are secret/internal fields tested as absent?
- Are nested object graphs finite?
- Is `many=True` used at every collection boundary?
- Could `attr=` replace unnecessary `MethodField` code?
- Are money/date/enum conversions consistent?
- Are optional and missing fields handled intentionally?
- Is JSON encoding kept separate from object serialization?
- Are serializer instances short-lived if the source can mutate?
- Are incoming payloads validated before Serpy sees domain objects?
- Do contract tests protect public output shape?
- Have performance claims been measured on realistic data?


# Suggested exercises without solutions

Use the patterns in this notebook to solve these independently.

1. **Paginated search results**  
   Serialize result objects with a separate top-level pagination envelope.

2. **Enum status field**  
   Create a custom field that emits `Enum.value`.

3. **Masked identifier**  
   Emit only the final four characters of a sensitive reference.

4. **Duration field**  
   Convert a `datetime.timedelta` to whole seconds.

5. **Localized serializer configuration**  
   Extend the serializer-aware custom getter example to add a locale-dependent prefix.

6. **Public vs staff serializers**  
   Design two explicit projections of the same user model.

7. **Event stream**  
   Serialize heterogeneous event payloads using a `MethodField` dispatch table.

8. **Performance investigation**  
   Compare a shallow serializer, a nested serializer, and a serializer containing expensive computed fields.

9. **Contract regression test**  
   Freeze a known JSON fixture and detect unintended field additions.

10. **Circular ORM relationship**  
    Design summary serializers that prevent recursive expansion.


# End

The central idea to keep is simple:

> A good serializer is not an object dump. It is an intentional boundary.

Serpy stays small, so much of the quality comes from how thoughtfully you choose that boundary.
